# 70+ Mobility 빠른 분석

## 배경
- 발표에서 70+ 검출률이 **1.5**로 관측됨 (실제 인구의 1.5배 검출).
- 추정 원인: **주민등록-실거주 mismatch** — 자녀 집에 거주하는 노인이 본가에는 등록만 남아있음. 분모(MOIS)가 분자(KT)보다 작아져 비율이 1을 초과.
- 본 노트북은 70+의 mobility **패턴 자체**를 빠르게 점검하여 **정적 모델 결정의 합리성**을 검증.

## 가설
1. 70+ 일일 이동량 자체는 적지 않을 수 있음 (검출률 1.5)
2. 그러나 **이동거리가 짧고 활동 반경이 좁음** → 행정동 간 이동은 작음
3. 시간대 패턴: 출퇴근 피크 없음, 종일 분산
4. 결론: 절대 이동량이 많아도 "같은 행정동 안" 위주라면 정적 모델 정당화 가능

In [ ]:
import os
import sys
import datetime as dt
from pathlib import Path

import polars as pl
import numpy as np
import matplotlib.pyplot as plt

_HERE = Path.cwd()
PROJECT_ROOT = _HERE.parent if _HERE.name == 'notebooks' else _HERE
os.chdir(PROJECT_ROOT)

DATA = PROJECT_ROOT / 'data/raw/movement/movement_sudogwon_202301.parquet'
OUT = PROJECT_ROOT / 'outputs/age_70plus_check'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 120
print('DATA:', DATA, 'exists:', DATA.exists())
print('OUT: ', OUT)

## 1. 연령별 일일 평균 이동량 (평일)

In [ ]:
df = pl.read_parquet(DATA)
print(f'Total rows: {len(df):,}')

HOLIDAYS_202301 = {20230101, 20230121, 20230122, 20230123, 20230124}

def is_weekday(yyyymmdd: int) -> bool:
    if yyyymmdd in HOLIDAYS_202301:
        return False
    d = dt.date(yyyymmdd // 10000, (yyyymmdd // 100) % 100, yyyymmdd % 100)
    return d.weekday() < 5

weekdays_only = df.filter(pl.col('date').map_elements(is_weekday, return_dtype=pl.Boolean))
n_weekdays = weekdays_only['date'].n_unique()
print(f'평일 일수: {n_weekdays}')
print(f'평일 행 수: {len(weekdays_only):,}')

by_age = (
    weekdays_only.group_by('age_10').agg([
        (pl.col('total').sum() / n_weekdays).alias('daily_avg_trips'),
        pl.col('total').sum().alias('total_trips'),
        pl.len().alias('n_records'),
    ]).sort('age_10')
)
print('\n연령대별 일일 평균 이동량 (평일):')
print(by_age)

AGE_LABELS = {0: '0-9', 10: '10-19', 20: '20-29', 30: '30-39',
              40: '40-49', 50: '50-59', 60: '60-69', 70: '70+'}
ages = by_age['age_10'].to_numpy()
labels = [AGE_LABELS.get(int(a), str(a)) for a in ages]
trips = by_age['daily_avg_trips'].to_numpy()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['C0'] * (len(ages) - 1) + ['C3']  # 마지막(70+) 강조
ax.bar(labels, trips, color=colors, alpha=0.8)
for i, v in enumerate(trips):
    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Age group')
ax.set_ylabel('Daily average trips')
ax.set_title('Daily average trips by age (2023-01 weekday)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT / '01_daily_trips_by_age.png', dpi=120)
plt.show()

## 2. 연령별 이동거리 분포

`avg_dist`는 m 단위. 같은 행정동 내 이동(짧은 거리)도 포함.

In [ ]:
dist_stats = (
    weekdays_only.group_by('age_10').agg([
        pl.col('avg_dist').quantile(0.10).alias('p10'),
        pl.col('avg_dist').quantile(0.25).alias('p25'),
        pl.col('avg_dist').median().alias('p50'),
        pl.col('avg_dist').quantile(0.75).alias('p75'),
        pl.col('avg_dist').quantile(0.90).alias('p90'),
        pl.col('avg_dist').mean().alias('mean'),
    ]).sort('age_10')
)
print('연령대별 avg_dist 분포 (m):')
print(dist_stats)

ages_list = sorted(weekdays_only['age_10'].unique().to_list())
data = []
for age in ages_list:
    arr = weekdays_only.filter(pl.col('age_10') == age)['avg_dist'].to_numpy()
    arr = arr[(arr > 0) & (arr < 50000)]
    data.append(arr)

fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot(
    data,
    labels=[AGE_LABELS.get(int(a), str(a)) for a in ages_list],
    showfliers=False,
    patch_artist=True,
)
for patch, age in zip(bp['boxes'], ages_list):
    patch.set_facecolor('C3' if age == 70 else 'C0')
    patch.set_alpha(0.6)
ax.set_xlabel('Age group')
ax.set_ylabel('avg_dist (m)')
ax.set_title('Trip distance distribution by age (2023-01 weekday, 0–50km)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / '02_distance_distribution.png', dpi=120)
plt.show()

## 3. 70+ vs 30–50대 직접 비교

성인 baseline(30–50대 통합) 대비 70+의 이동 패턴 차이를 살펴봄.

In [ ]:
adult_3050 = weekdays_only.filter(pl.col('age_10').is_in([30, 40, 50]))
elderly_70 = weekdays_only.filter(pl.col('age_10') == 70)

print(f'30-50대 평일 이동 행 수: {len(adult_3050):,}')
print(f'70+ 평일 이동 행 수:    {len(elderly_70):,}')

def same_dong_ratio(d):
    same = d.filter(pl.col('o_admdong_cd') == pl.col('d_admdong_cd'))['total'].sum()
    total = d['total'].sum()
    return same / total * 100

print(f'\n30-50대 같은 동 내 이동 비율: {same_dong_ratio(adult_3050):.1f}%')
print(f'70+    같은 동 내 이동 비율: {same_dong_ratio(elderly_70):.1f}%')

# 이동거리 히스토그램 (정규화)
adult_dist = adult_3050.filter((pl.col('avg_dist') > 0) & (pl.col('avg_dist') < 20000))['avg_dist'].to_numpy()
elderly_dist = elderly_70.filter((pl.col('avg_dist') > 0) & (pl.col('avg_dist') < 20000))['avg_dist'].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(adult_dist, bins=50, alpha=0.5, label='30-50대', density=True, color='C0')
axes[0].hist(elderly_dist, bins=50, alpha=0.5, label='70+', density=True, color='C3')
axes[0].set_xlabel('Trip distance (m)')
axes[0].set_ylabel('Density')
axes[0].set_title('Trip distance: 30-50대 vs 70+ (0–20km)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 시간대별 비교
hourly_adult = adult_3050.group_by('hour').agg(pl.col('total').sum().alias('trips')).sort('hour')
hourly_elderly = elderly_70.group_by('hour').agg(pl.col('total').sum().alias('trips')).sort('hour')
adult_norm = hourly_adult['trips'].to_numpy() / hourly_adult['trips'].sum() * 100
elderly_norm = hourly_elderly['trips'].to_numpy() / hourly_elderly['trips'].sum() * 100

axes[1].plot(hourly_adult['hour'], adult_norm, marker='o', label='30-50대', color='C0')
axes[1].plot(hourly_elderly['hour'], elderly_norm, marker='s', label='70+', color='C3')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('% of daily trips')
axes[1].set_title('Hourly mobility: 30-50대 vs 70+')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / '03_compare_30to50_vs_70.png', dpi=120)
plt.show()

## 4. 70+ 활동 반경 — 출발 행정동 당 도착 행정동 수

활동 반경(geographic spread)을 도착 행정동 unique count로 측정.
"좁다"의 정량적 정의: 30–50대 대비 적은 destination 수.

In [ ]:
od_count_70 = (
    elderly_70.group_by('o_admdong_cd').agg([
        pl.col('d_admdong_cd').n_unique().alias('n_destinations'),
        pl.col('total').sum().alias('total_trips'),
    ])
)
od_count_adult = (
    adult_3050.group_by('o_admdong_cd').agg([
        pl.col('d_admdong_cd').n_unique().alias('n_destinations'),
        pl.col('total').sum().alias('total_trips'),
    ])
)

print('=== 출발 행정동당 평균 도착 행정동 수 ===')
print(f'30-50대: mean={od_count_adult["n_destinations"].mean():.1f}  median={od_count_adult["n_destinations"].median():.0f}')
print(f'70+:     mean={od_count_70["n_destinations"].mean():.1f}  median={od_count_70["n_destinations"].median():.0f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(od_count_adult['n_destinations'].to_numpy(), bins=30, alpha=0.5,
        label='30-50대', density=True, color='C0')
ax.hist(od_count_70['n_destinations'].to_numpy(), bins=30, alpha=0.5,
        label='70+', density=True, color='C3')
ax.set_xlabel('# unique destinations per origin admdong')
ax.set_ylabel('Density')
ax.set_title('Activity range — destinations per origin admdong (weekday Jan 2023)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / '04_activity_range.png', dpi=120)
plt.show()

## 5. 결론

본 셀 실행 후 채울 부분:

| 지표 | 30–50대 | 70+ | 해석 |
|---|---|---|---|
| 일일 평균 trips |  |  | (절대 이동량) |
| 같은 동 비율 |  |  | 높을수록 좁은 반경 |
| 중앙 trip 거리 |  |  | (m) |
| 도착 행정동 수 (mean) |  |  | (활동 반경) |
| 출퇴근 피크 |  |  | 8/18시 봉우리 유무 |

**정적 모델 결정의 합리성**
- 70+의 절대 이동량은 검출률 1.5 영향으로 작지 않게 보일 수 있음.
- 그러나 **같은 동 내 이동 비율↑ + 도착 행정동 수↓ + 중앙 거리↓** 라면, 행정동 간 mobility 신호는 약함.
- ∴ "70+ = 자기 행정동 내 정적" 가정이 모델 동학에 미치는 영향 작음 → 정당화.

**다음 단계**
- 학기 중(2023-05/10) 데이터로 재검증
- 70+의 검출률 1.5 원인을 행정동 단위로 분해 (도심/외곽 패턴)